In [2]:
import sys
sys.path.append('..')

from pypdf import PdfReader
from model import Chunk
from src.adapters.openrouter import OpenRouterService
from src.adapters.chroma_db import ChromaDBService

In [3]:
PDF_PATH = "../data/NIPS-2017-attention-is-all-you-need-Paper.pdf"

reader = PdfReader(PDF_PATH)

full_text = ''
for page_number, page in enumerate(reader.pages):
    text = page.extract_text()
    full_text += text + "\n"

print(f"Total pages: {len(reader.pages)}")
print(f"Total characters extracted: {len(full_text)}")
print(full_text[:500])

Total pages: 11
Total characters extracted: 32624
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or


In [4]:
def chunk_text(text: str, chunk_size: int=800, overlap: int=150) -> list[str]:
    """
    Splits the text into overlapping chunks.
    chunk_size: the number of characters in each chunk
    overlap: the amount of overlap between consecutive chunks (to preserve context)

    """
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start += chunk_size - overlap  

    return [c for c in chunks if c]  # Remove empty chunks


raw_chunks = chunk_text(full_text)
print(f"Total chunks created: {len(raw_chunks)}")
print(raw_chunks[0])

Total chunks created: 51
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and co


In [5]:
document_name = "attention_is_all_you_need"

chunks = [
    Chunk(
        id=f"{document_name}_chunk_{i}",
        text=chunk_content,
        metadata={"source": document_name, "chunk_index": i}
    )
    for i, chunk_content in enumerate(raw_chunks)
]

print(f"Total Chunk objects: {len(chunks)}")
print(chunks[0])

Total Chunk objects: 51
Chunk(id='attention_is_all_you_need_chunk_0', text='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence and co', metadata={'source': 'attention_is_all_you_need', 'chunk_index': 0})


In [6]:
embedder = OpenRouterService()
db = ChromaDBService(embedder=embedder, collection_name="attention_paper")

db.add_document(chunks)
print("All chunks embedded and stored in ChromaDB!")

All chunks embedded and stored in ChromaDB!


In [7]:
results = db.fetch_chunks("What is the transformer architecture?", top_k=3)

for r in results:
    print(f"[{r.id}] distance={r.distance:.4f}")
    print(r.text[:200])
    print("---")

[attention_is_all_you_need_chunk_11] distance=0.8733
tion-
2
Figure 1: The Transformer - model architecture.
wise fully connected feed-forward network. We employ a residual connection [10] around each of
the two sub-layers, followed by layer normalizati
---
[attention_is_all_you_need_chunk_9] distance=0.9338
e best of our knowledge, however, the Transformer is the ﬁrst transduction model relying
entirely on self-attention to compute representations of its input and output without using sequence-
aligned R
---
[attention_is_all_you_need_chunk_10] distance=0.9422
er then generates an output
sequence (y1,...,y m) of symbols one element at a time. At each step the model is auto-regressive
[9], consuming the previously generated symbols as additional input when g
---
